# Text-to-SQL Demo with Medha and DuckDB

This notebook walks through a complete Text-to-SQL workflow using **DuckDB** — an
in-process OLAP database designed for analytical queries.

1. Create an in-memory DuckDB database with sample data
2. Store question-SQL pairs in Medha's semantic cache
3. Demonstrate all four cache tiers:
   - **Tier 0** — L1 in-memory cache (microseconds)
   - **Tier 1** — Template matching with parameter extraction
   - **Tier 2** — Exact vector match
   - **Tier 3** — Semantic similarity match

**Requirements:** `pip install "medha-archai[fastembed]" duckdb`

## Cell 1: Setup & Imports

In [ ]:
import time

import duckdb

from medha import Medha, Settings
from medha.embeddings.fastembed_adapter import FastEmbedAdapter
from medha.types import QueryTemplate

## Cell 2: Create a DuckDB Database

We create an in-memory DuckDB database with two tables:
- `sales` — 8 rows with region, product, amount, and quarter
- `customers` — 5 rows with name, country, and tier

DuckDB shines on analytical workloads — aggregations, window functions, and
columnar scans — making it a natural fit for Text-to-SQL AI assistants.

In [ ]:
conn = duckdb.connect(":memory:")

conn.execute("""
    CREATE TABLE sales (
        id      INTEGER,
        region  VARCHAR,
        product VARCHAR,
        amount  DOUBLE,
        quarter VARCHAR
    );
    INSERT INTO sales VALUES
        (1, 'North', 'Widget A', 12000, 'Q1'),
        (2, 'South', 'Widget B', 8500,  'Q1'),
        (3, 'North', 'Widget A', 15000, 'Q2'),
        (4, 'East',  'Gizmo X',  22000, 'Q2'),
        (5, 'West',  'Widget B', 9800,  'Q3'),
        (6, 'East',  'Gizmo X',  18000, 'Q3'),
        (7, 'North', 'Widget A', 17500, 'Q4'),
        (8, 'South', 'Gizmo X',  14000, 'Q4');

    CREATE TABLE customers (
        id      INTEGER,
        name    VARCHAR,
        country VARCHAR,
        tier    VARCHAR
    );
    INSERT INTO customers VALUES
        (1, 'Acme Corp',    'USA',    'Gold'),
        (2, 'Globex Inc',   'UK',     'Silver'),
        (3, 'Initech Ltd',  'Canada', 'Gold'),
        (4, 'Umbrella Co',  'USA',    'Bronze'),
        (5, 'Hooli GmbH',   'Germany','Silver');
""")

print("DuckDB database created with 'sales' and 'customers' tables.")
print("Sales rows:", conn.execute("SELECT COUNT(*) FROM sales").fetchone()[0])
print("Customer rows:", conn.execute("SELECT COUNT(*) FROM customers").fetchone()[0])

## Cell 3: Initialize Medha

We use:
- **FastEmbedAdapter** — local embeddings, no API key needed
- **`backend_type="qdrant"`** — Qdrant backend (default); use `"memory"` for zero-deps
- **`qdrant_mode="memory"`** — Qdrant runs in-process, no external server required
- **`query_language="sql"`** — labels the cache as SQL-oriented

In [ ]:
embedder = FastEmbedAdapter(model_name="BAAI/bge-small-en-v1.5", cache_dir="./.fastembed_cache")
settings = Settings(
    backend_type="qdrant",      # "memory" = pure-Python (zero deps), "pgvector" = PostgreSQL
    qdrant_mode="memory",
    query_language="sql",
)

medha = Medha(collection_name="duckdb_demo", embedder=embedder, settings=settings)
await medha.start()
print("Medha initialized (collection='duckdb_demo', backend='qdrant/memory', language='sql')")

## Cell 4: Store Question-SQL Pairs

We store 7 question-SQL pairs. Each SQL query leverages DuckDB's analytical
capabilities — aggregations, window functions, and GROUP BY — and the actual
result is stored as `response_summary`.

In [ ]:
pairs = [
    (
        "What is the total revenue?",
        "SELECT SUM(amount) AS total_revenue FROM sales",
    ),
    (
        "Show revenue by region",
        "SELECT region, SUM(amount) AS revenue FROM sales GROUP BY region ORDER BY revenue DESC",
    ),
    (
        "Which product generated the most sales?",
        "SELECT product, SUM(amount) AS total FROM sales GROUP BY product ORDER BY total DESC LIMIT 1",
    ),
    (
        "Show me quarterly revenue trends",
        "SELECT quarter, SUM(amount) AS revenue FROM sales GROUP BY quarter ORDER BY quarter",
    ),
    (
        "How many Gold tier customers do we have?",
        "SELECT COUNT(*) FROM customers WHERE tier = 'Gold'",
    ),
    (
        "List all customers from the USA",
        "SELECT name, tier FROM customers WHERE country = 'USA'",
    ),
    (
        "What is the average sale amount per quarter?",
        "SELECT quarter, AVG(amount) AS avg_sale FROM sales GROUP BY quarter ORDER BY quarter",
    ),
]

for question, sql in pairs:
    result = conn.execute(sql).fetchall()
    await medha.store(question, sql, response_summary=str(result))
    print(f"Stored: {question!r}")
    print(f"  SQL:    {sql}")
    print(f"  Result: {result}\n")

## Cell 5: Tier 2 — Exact Vector Match

When we search with the **exact same question** that was stored, Medha finds an
exact vector match (cosine similarity ~0.99+). This is Tier 2 in the waterfall.

In [ ]:
start = time.perf_counter()
hit = await medha.search("What is the total revenue?")
elapsed = (time.perf_counter() - start) * 1000

print(f"Strategy: {hit.strategy}")
print(f"Query:    {hit.generated_query}")
print(f"Score:    {hit.confidence:.4f}")
print(f"Time:     {elapsed:.2f}ms")

# Execute against DuckDB to confirm it works
print(f"\nDuckDB result: {conn.execute(hit.generated_query).fetchone()}")

## Cell 6: Tier 3 — Semantic Similarity Match

Now we search with a **rephrased question** that was never stored. Medha recognizes
the semantic intent and returns the matching SQL from the cache.

In [ ]:
queries = [
    ("What's our overall revenue figure?",         "What is the total revenue?"),
    ("Break down sales amounts by geographic area", "Show revenue by region"),
    ("Which item sold the most?",                  "Which product generated the most sales?"),
]

for question, expected_match in queries:
    start = time.perf_counter()
    hit = await medha.search(question)
    elapsed = (time.perf_counter() - start) * 1000

    print(f"Question: {question!r}")
    print(f"  Strategy: {hit.strategy}")
    print(f"  Query:    {hit.generated_query}")
    print(f"  Score:    {hit.confidence:.4f}  ({elapsed:.2f}ms)")
    print()

## Cell 7: Tier 1 — Template Matching with Parameter Extraction

Templates define parameterized patterns. Here we define two templates:
- Filter sales by quarter (`Q1`–`Q4`)
- Filter customers by tier (`Gold`, `Silver`, `Bronze`)

When a question matches, Medha extracts the parameter via regex and fills it
into the query template — no LLM call needed.

In [ ]:
templates = [
    QueryTemplate(
        intent="sales_by_quarter",
        template_text="Show sales for {quarter}",
        query_template="SELECT region, product, amount FROM sales WHERE quarter = '{quarter}'",
        parameters=["quarter"],
        parameter_patterns={"quarter": r"\b(Q[1-4])\b"},
    ),
    QueryTemplate(
        intent="customers_by_tier",
        template_text="List {tier} customers",
        query_template="SELECT name, country FROM customers WHERE tier = '{tier}'",
        parameters=["tier"],
        parameter_patterns={"tier": r"\b(Gold|Silver|Bronze)\b"},
    ),
]

await medha.load_templates(templates)
print(f"Loaded {len(templates)} template(s)\n")

test_questions = [
    "Show sales for Q3",
    "Give me all Silver customers",
]

for question in test_questions:
    start = time.perf_counter()
    hit = await medha.search(question)
    elapsed = (time.perf_counter() - start) * 1000

    print(f"Question: {question!r}")
    print(f"  Strategy: {hit.strategy}")
    print(f"  Query:    {hit.generated_query}")
    print(f"  Time:     {elapsed:.2f}ms")

    if hit.generated_query:
        result = conn.execute(hit.generated_query).fetchall()
        print(f"  Results:  {result}\n")
    else:
        print(f"  Results:  [no query — template match failed]\n")


## Cell 8: Tier 0 — L1 In-Memory Cache

The L1 cache stores recent search results in memory. The first call goes through
the full waterfall. The second call for the **same question** returns instantly
from L1 — typically >100x faster.

In [ ]:
# First call — full waterfall
start = time.perf_counter()
hit1 = await medha.search("Show revenue by region")
t1 = (time.perf_counter() - start) * 1000

# Second call — served from L1 cache
start = time.perf_counter()
hit2 = await medha.search("Show revenue by region")
t2 = (time.perf_counter() - start) * 1000

print(f"First call:  {t1:.2f}ms  ({hit1.strategy})")
print(f"Second call: {t2:.4f}ms ({hit2.strategy})")
if t2 > 0:
    print(f"Speedup:     {t1 / t2:.0f}x")

## Cell 9: DuckDB Window Function Query

DuckDB natively supports SQL window functions. Here we cache a query that uses
`SUM(...) OVER (PARTITION BY ...)` to compute a running regional share — a query
type that would be expensive to regenerate repeatedly via an LLM.

In [ ]:
window_sql = """
SELECT
    region,
    quarter,
    amount,
    SUM(amount) OVER (PARTITION BY region ORDER BY quarter) AS running_total
FROM sales
ORDER BY region, quarter
""".strip()

result = conn.execute(window_sql).fetchall()
await medha.store(
    "What is the running revenue total per region across quarters?",
    window_sql,
    response_summary=str(result),
)
print("Stored window-function query.\n")

# Retrieve with a semantic paraphrase
hit = await medha.search("Show me cumulative sales by region over time")
print(f"Strategy: {hit.strategy}")
print(f"Score:    {hit.confidence:.4f}")
print(f"Query:\n{hit.generated_query}")
print(f"\nDuckDB result (first 5 rows): {conn.execute(hit.generated_query).fetchmany(5)}")

## Cell 10: Stats & Cleanup

In [ ]:
print("Cache Statistics:")
print(medha.stats)

await medha.close()
conn.close()
print("\nCleaned up: Medha closed, DuckDB connection closed.")